### FAISS

##### Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

text_loader = TextLoader('speech.txt')
documents = text_loader.load()

text_splitter = CharacterTextSplitter(separator= "\n\n",chunk_size=1000,chunk_overlap=30)
docs = text_splitter.split_documents(documents)
print(docs)
print("---------------------------")
print(docs[0])


Created a chunk of size 1117, which is longer than the specified 1000


[Document(metadata={'source': 'speech.txt'}, page_content="Five score years ago, a great American, in whose symbolic shadow we stand today, signed the Emancipation Proclamation. This momentous decree came as a great beacon light of hope to millions of Negro slaves who had been seared in the flames of withering injustice. It came as a joyous daybreak to end the long night of their captivity.\n\nBut one hundred years later, the Negro still is not free. One hundred years later, the life of the Negro is still sadly crippled by the manacles of segregation and the chains of discrimination. One hundred years later, the Negro lives on a lonely island of poverty in the midst of a vast ocean of material prosperity. One hundred years later, the Negro is still languished in the corners of American society and finds himself an exile in his own land. And so we've come here today to dramatize a shameful condition."), Document(metadata={'source': 'speech.txt'}, page_content="In a sense we've come to o

In [10]:
import os # Our API key is stored in environment variables
from dotenv import load_dotenv # To load environment variables from a .env file
load_dotenv() # To Load All Environment variables from .env file


# API Key loading from .env file
os.environ["HF_API_KEY"] = os.getenv("HF_API_KEY")

In [11]:
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(docs,hf_embeddings)
db

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10084.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
### Querying

query = "I have a dream that one day this"

docs = db.similarity_search(query)
docs

[Document(id='eee79e0a-f406-4289-9a8b-50158cdb7b40', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day this nation will rise up and live out the true meaning of its creed: 'We hold these truths to be self-evident, that all men are created equal.'\n\nI have a dream that one day on the red hills of Georgia, the sons of former slaves and the sons of former slave owners will be able to sit down together at the table of brotherhood.\n\nI have a dream that one day even the state of Mississippi, a state sweltering with the heat of injustice, sweltering with the heat of oppression, will be transformed into an oasis of freedom and justice.\n\nI have a dream that my four little children will one day live in a nation where they will not be judged by the color of their skin but by the content of their character.\n\nI have a dream today!"),
 Document(id='3858e406-0300-4d8f-83ad-190357196e37', metadata={'source': 'speech.txt'}, page_content="I have a dream today!\n\nI have

In [16]:
docs[0].page_content

"I have a dream that one day this nation will rise up and live out the true meaning of its creed: 'We hold these truths to be self-evident, that all men are created equal.'\n\nI have a dream that one day on the red hills of Georgia, the sons of former slaves and the sons of former slave owners will be able to sit down together at the table of brotherhood.\n\nI have a dream that one day even the state of Mississippi, a state sweltering with the heat of injustice, sweltering with the heat of oppression, will be transformed into an oasis of freedom and justice.\n\nI have a dream that my four little children will one day live in a nation where they will not be judged by the color of their skin but by the content of their character.\n\nI have a dream today!"

### As a Retriever

#### We can also convert the vector store into Retriever class. This Allows us to easily use it in other langchain methods, Which largely work with retrievers

In [19]:
retriever = db.as_retriever()
docs_new = retriever.invoke(query)

In [20]:
docs_new[0].page_content

"I have a dream that one day this nation will rise up and live out the true meaning of its creed: 'We hold these truths to be self-evident, that all men are created equal.'\n\nI have a dream that one day on the red hills of Georgia, the sons of former slaves and the sons of former slave owners will be able to sit down together at the table of brotherhood.\n\nI have a dream that one day even the state of Mississippi, a state sweltering with the heat of injustice, sweltering with the heat of oppression, will be transformed into an oasis of freedom and justice.\n\nI have a dream that my four little children will one day live in a nation where they will not be judged by the color of their skin but by the content of their character.\n\nI have a dream today!"

### Similarity Search with Score

#### There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only documents but also the distance score of the query to them. The returned distance score is L2(manhattan distance) distance. Therefore, a lower score is better

In [21]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='eee79e0a-f406-4289-9a8b-50158cdb7b40', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day this nation will rise up and live out the true meaning of its creed: 'We hold these truths to be self-evident, that all men are created equal.'\n\nI have a dream that one day on the red hills of Georgia, the sons of former slaves and the sons of former slave owners will be able to sit down together at the table of brotherhood.\n\nI have a dream that one day even the state of Mississippi, a state sweltering with the heat of injustice, sweltering with the heat of oppression, will be transformed into an oasis of freedom and justice.\n\nI have a dream that my four little children will one day live in a nation where they will not be judged by the color of their skin but by the content of their character.\n\nI have a dream today!"),
  np.float32(0.89977115)),
 (Document(id='3858e406-0300-4d8f-83ad-190357196e37', metadata={'source': 'speech.txt'}, page_content="I 

In [22]:
embedding_vector = hf_embeddings.embed_query(query)
embedding_vector

[0.04876967892050743,
 -0.04109163582324982,
 0.020216934382915497,
 0.0313822403550148,
 -0.05802789703011513,
 -0.08306944370269775,
 0.0582815483212471,
 -0.03990495577454567,
 0.1042829230427742,
 -0.02876664698123932,
 -0.06296693533658981,
 0.009717590175569057,
 0.015680968761444092,
 -0.016484269872307777,
 0.03577231243252754,
 -0.03114785999059677,
 -0.028054052963852882,
 -0.024670781567692757,
 -0.04179070517420769,
 0.012660721316933632,
 -0.04649902880191803,
 0.0024243020452558994,
 0.01734287105500698,
 -0.003442582907155156,
 -0.06005249544978142,
 0.05062904581427574,
 -0.01989453285932541,
 -0.010120890103280544,
 0.004932694137096405,
 -0.02349395863711834,
 0.047122079879045486,
 -0.060116108506917953,
 -0.0460311621427536,
 0.06786985695362091,
 0.06654957681894302,
 -0.00273284874856472,
 -0.01578647643327713,
 -0.007819580845534801,
 0.01857125572860241,
 0.02667844481766224,
 0.016693247482180595,
 -0.017573710530996323,
 0.01730821467936039,
 -0.00829621590673

In [23]:
docs_and_score_new = db.similarity_search_by_vector(embedding_vector)
docs_and_score_new

[Document(id='eee79e0a-f406-4289-9a8b-50158cdb7b40', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day this nation will rise up and live out the true meaning of its creed: 'We hold these truths to be self-evident, that all men are created equal.'\n\nI have a dream that one day on the red hills of Georgia, the sons of former slaves and the sons of former slave owners will be able to sit down together at the table of brotherhood.\n\nI have a dream that one day even the state of Mississippi, a state sweltering with the heat of injustice, sweltering with the heat of oppression, will be transformed into an oasis of freedom and justice.\n\nI have a dream that my four little children will one day live in a nation where they will not be judged by the color of their skin but by the content of their character.\n\nI have a dream today!"),
 Document(id='3858e406-0300-4d8f-83ad-190357196e37', metadata={'source': 'speech.txt'}, page_content="I have a dream today!\n\nI have

In [24]:
### Saving and loading the file in our local system

db.save_local("faiss_index")

In [ ]:
## To load the above stored local file

new_df = FAISS.load_local("faiss_index",hf_embeddings)

In [26]:
## To load the above stored local file

new_df = FAISS.load_local("faiss_index",hf_embeddings,allow_dangerous_deserialization = True)
new_df.similarity_search(query)

[Document(id='eee79e0a-f406-4289-9a8b-50158cdb7b40', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day this nation will rise up and live out the true meaning of its creed: 'We hold these truths to be self-evident, that all men are created equal.'\n\nI have a dream that one day on the red hills of Georgia, the sons of former slaves and the sons of former slave owners will be able to sit down together at the table of brotherhood.\n\nI have a dream that one day even the state of Mississippi, a state sweltering with the heat of injustice, sweltering with the heat of oppression, will be transformed into an oasis of freedom and justice.\n\nI have a dream that my four little children will one day live in a nation where they will not be judged by the color of their skin but by the content of their character.\n\nI have a dream today!"),
 Document(id='3858e406-0300-4d8f-83ad-190357196e37', metadata={'source': 'speech.txt'}, page_content="I have a dream today!\n\nI have